# Run Qwen3.6 27B `llama.cpp`

[llama.cpp](https://github.com/ggml-org/llama.cpp) was originally conceived as a software
to run quantized LLMs on the CPU. In the meantime, it supports also major GPU
architectures.

Compared to `vLLM`, `llama.cpp` uses another data format called `GGUF`. The `GGUF` files
are also hosted on Hugging Face and `llama.cpp` integrates into that infrastructure.

Why should you use `llama.cpp`? It is very efficient and `GGUF` files are often available
in many different quantization levels. This means that you can choos a `GGUF` that
just fits your GPU (or CPU) RAM.

`llama.cpp` both has a frontend, but can also act as an Open AI compatible API. This is
how we use it in this notebook.

First, run `llama-server -hf unsloth/Qwen3.5-27B-GGUF:UD-Q4_K_XL --port 8000 --host 0.0.0.0`

In [ ]:
!nvidia-smi

vLLM offers an Open AI compatible API:

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="secret")

We want to use the notebook for different models:

In [ ]:
model = "unsloth/Qwen3.5-27B-GGUF:UD-Q4_K_XL"

## Chat Completion API

Use the chat completion API first:

In [ ]:
completion = client.chat.completions.create(
    model=model, 
    messages=[{ "role": "user",
                "content": "How many 'r's are in 'strawberry'?" } ]
)

print(completion.choices[0].message.content)

The message does not consist of `content` only:

In [ ]:
completion.choices[0].message

In [ ]:
from IPython.display import display, Markdown
# note the different accessor:
display(Markdown(completion.choices[0].message.reasoning_content))

In [ ]:
display(Markdown(completion.choices[0].message.content))

## Responses API

Now switch to the more modern responses API

In [ ]:
response = client.responses.create(model=model, 
                                   input="How many 'r's are in 'strawberry'?")

display(Markdown(response.output_text))

Examine the response:

In [ ]:
response

In [ ]:
display(Markdown(response.output[0].content[0].text))

In [ ]:
display(Markdown(response.output[1].content[0].text))

Can we stop the reasoning process for a specific question?

In [ ]:
response = client.responses.create(model=model, 
                                   input="How many 'r's are in 'strawberry'?",
                                   extra_body={ "chat_template_kwargs": {"enable_thinking": False} } )

response.output

In [ ]:
display(Markdown(response.output_text))

Compared to `vLLM`, this works directly in `llama.cpp`. As you can see, the API is still not fully consistent.